###### Qiskit dynamics error 때문에 파이썬 시뮬레이션 불가. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import qiskit
from qiskit.exceptions import QiskitError

#from qiskit import QuantumCircuit, QuantumRegister
#from qiskit_aer import AerSimulator
from qiskit_dynamics import LindbladModel

: 

In [2]:
print(f"Qiskit version: {qiskit.__version__}")

AttributeError: module 'qiskit' has no attribute '__version__'

Construct states
(res_M 구현 잘못함. 원래 normalized된 성분을 사용하려고 했는데, 저렇게 쓰면 그냥 identitiy가 나옴)

In [ ]:
sys_M = 1*np.identity(2)
res_M = (1/np.sqrt(2*np.identity(2)[0,0]**2 + 2*np.identity(2)[1,1]**2))*2*np.identity(2)
p_M = 3*np.identity(2)
q_M = 4*np.identity(2)
h_M = 5*np.identity(2)

# 순서 맞춰서 입력해야 함
rho_test = np.kron(sys_M,np.kron(res_M,np.kron(h_M,np.kron(p_M,q_M))))

In [21]:
rho_test

array([[60.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0., 60.,  0., ...,  0.,  0.,  0.],
       [ 0.,  0., 60., ...,  0.,  0.,  0.],
       ...,
       [ 0.,  0.,  0., ..., 60.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0., 60.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0., 60.]])

Operator construction
##### 여기 (i>>4) & 1 부분 왜 이렇게 구현되는지 잘 모르겠음. 꼭 다시 생각해볼 것.

In [1]:
def hamiltonian_OP():
    """
    5-큐비트(s,r,h,p,q) 공간에서 s와 h 큐비트를 SWAP하는
    32x32 유니터리 행렬을 생성합니다.
    """
    dim = 2**5
    swap_matrix = np.zeros((dim, dim), dtype=complex)

    for i in range(dim):
        # 사용자 정의 순서 |s,r,h,p,q⟩에 따라 비트 값 추출
        s = (i >> 4) & 1 # 이거 무슨 의미인지 잘 모르겠음. 일단 돌려놓고 좀 물어봐야 할 것 같은데... 
        r = (i >> 3) & 1
        h = (i >> 2) & 1
        p = (i >> 1) & 1
        q = (i >> 0) & 1

        # s와 h의 값을 바꾼 출력 기저 상태 |h,r,s,p,q⟩를 만듭니다.
        j = (h << 4) | (r << 3) | (s << 2) | (p << 1) | (q << 0)

        # 변환 규칙에 따라 행렬의 [j, i] 위치에 1을 설정합니다.
        swap_matrix[j, i] = 1
        
    return swap_matrix

In [3]:
def lindbladian_OP():
    """
    5-큐비트(S,R,H,P,Q) 공간에서 M 연산자의 32x32 행렬을 생성합니다.
    M = (1/sqrt(d)) * (I_SRH ⊗ |Γ⟩⟨Γ|_PQ) * (SWAP_SP ⊗ I_RHQ)
    """
    dim = 2**5
    d_system = 2  # 시스템 S의 차원
    
    # --- 1. SWAP(S, P) 연산자 행렬 생성 ---
    # 순서: S(4), R(3), H(2), P(1), Q(0)
    swap_sp_matrix = np.zeros((dim, dim), dtype=complex)
    s_pos, p_pos = 4, 1 # S와 P의 비트 위치
    
    for i in range(dim):
        s_bit = (i >> s_pos) & 1
        p_bit = (i >> p_pos) & 1
        
        # s와 p 비트가 다를 경우에만 위치를 교환
        if s_bit != p_bit:
            mask = (1 << s_pos) | (1 << p_pos)
            j = i ^ mask
        else:
            j = i
        swap_sp_matrix[j, i] = 1

    # --- 2. 얽힘 상태 프로젝터 연산자 행렬 생성 ---
    # |Γ⟩⟨Γ|_PQ 부분 (4x4 행렬)
    psi_gamma = np.zeros(d_system**2)
    for i in range(d_system):
        psi_gamma[i * d_system + i] = 1 / np.sqrt(d_system)
    projector_pq = np.outer(psi_gamma, psi_gamma.conj())
    
    # I_SRH 부분 (8x8 항등 행렬)
    I_srh = np.eye(2**3)
    
    # 크로네커 곱으로 전체 32x32 프로젝터 행렬 구성
    projector_full_matrix = np.kron(I_srh, projector_pq)

    # --- 3. 최종 M 행렬 계산 ---
    # M = (스칼라) * (프로젝터 부분) @ (SWAP 부분)
    M_matrix = (1 / np.sqrt(d_system)) * projector_full_matrix @ swap_sp_matrix
    
    return M_matrix

In [ ]:
H_hat_matrix = np.random.rand(32, 32) 
M_op_matrix = np.random.rand(32, 32)

# --- LindbladModel 객체 생성 ---

# H 행렬은 static_hamiltonian 인자에 전달합니다.
# M 행렬은 jump_operators 인자에 "리스트" 형태로 전달합니다. ⚙️
lindblad_system = LindbladModel(
    static_hamiltonian=H_hat_matrix,
    jump_operators=[M_op_matrix]
)

# 생성된 모델 객체를 출력하여 확인합니다.
print("LindbladModel 객체를 성공적으로 생성했습니다:")
print(lindblad_system)